# Spatial | Egocentric Head Direction Between Two Mice
-----------------------------------------

## Setup and Configuration

Before running this notebook, analyse the full video in the `DEEPLABCUT` environment with automatic tracking for two mice and CSV export enabled. The final stitched file must be saved in this recording's `VideoData` folder and end in `_el.csv`. Training labels and evaluation predictions are not full-video tracks.

In [1]:
##### Imports ##########################################################################

#==== Generic Imports ================
from pathlib import Path
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display


#==== Movement Imports ===============
from movement.io import load_poses
import movement.kinematics
import movement.utils.vector


#==== DataConduit Imports ============
from data_conduit.datastructures import slice_stream, slice_stream_per_trial


#==== Figure-Project Imports =========
from movement_figures.data_template.loading import build_datastructure
from movement_figures.misc import visual_streams_check, configure_simplified_trial_df
from movement_figures.timeseries_template.annotations import (
    AnnotationResult,
    annotate_qc_timeseries,
)


##### Notebook Settings ################################################################

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [2]:
##### Setup Parameters #################################################################

#==== Select the Recording ===========
ROOT = Path('/media/sepi/Elements/Video_example_2mice/2mice')         # One recording folder; its parent contains the sessions.


#==== Select the Two Mice ============
FOCAL_MOUSE = 'Mouse1'                                            # Mouse whose head direction defines the reference frame.
OTHER_MOUSE = 'Mouse2'                                            # Mouse used as the moving target.


In [3]:
##### Load DataStructure ###############################################################

#==== Load Trials and Video Times =======================================================
fig_ds_obj = build_datastructure(
    root        = ROOT.parent,                                   # Directory directly containing the recording folder.
    level_names = (),                                            # This example has no intermediate mouse/day folders.
    include     = (ROOT.name,),                                  # Load only the two-mouse recording.
    streams     = ('trials', 'video'),                            # Load multi-animal pose separately below.
    raw_events  = False,
)

fig_ds_obj.select()
streams = fig_ds_obj.load()


#==== Read the Two-Mouse Tracking Table =================================================
POSE_FILE = ROOT / 'DLC' / (
    'VideoData_1904-01-01T01-00-00DLC_DlcrnetStride16Ms5_2miceSep28shuffle1_'
    'snapshot_best-200_el_filtered.csv'
)

if not POSE_FILE.exists():
    raise FileNotFoundError(f'Pose file not found: {POSE_FILE}')

pose_table = pd.read_csv(
    filepath_or_buffer = POSE_FILE,
    header             = [0, 1, 2, 3],
    index_col          = 0,
)


#==== Align Pose to the Acquired Video Times ============================================
frame_times = streams['video'].index.to_numpy()
frame_numbers = range(len(frame_times))

if not pose_table.index.is_unique or not pose_table.index.isin(frame_numbers).all():
    raise ValueError('Pose rows must have unique, zero-based frame numbers within this video.')

pose_table = pose_table.reindex(frame_numbers)                    # Missing frames remain NaN; later timestamps do not shift.

recording = load_poses.from_dlc_style_df(
    df = pose_table,                                             # Keep both individual labels from the DLC export.
)
recording = recording.assign_coords(
    time    = frame_times,                                       # Use acquired seconds rather than reconstructing time from FPS.
    session = ('time', [ROOT.name] * len(frame_times)),
)
recording.time.attrs['units'] = 'seconds'
recording.attrs['time_unit'] = 'seconds'
streams['dlc:movement'] = recording


#==== Create the Compact Trial Table ===================================================
streams['compact_trials'] = configure_simplified_trial_df(
    trial_df        = streams['trials'],
    modify_in_place = False,
)


#==== Visual Streams Check =============================================================
visual_streams_check(
    loaded_ds_object   = streams,
    show_stream_shapes = True,
    show_stream_dtypes = True,
    display_streams    = False,
)


TypeError: VideoData times must contain real numeric acquired seconds.

## Select Trials to Show

In [ ]:
##### Trial Selection ##################################################################

#==== Extract Trial Range ============
selected_trials = slice_stream(streams['trials'][:8])              # Select up to eight trials; this recording contains five.


#==== Extract Recording Identifier ===
sessionID = selected_trials['session'].unique().item()


#==== Select Tracked Positions =======
recording = slice_stream(
    stream    = streams['dlc:movement'],
    selectors = {'session': sessionID},
)

position = recording['position'].sel(
    individual = FOCAL_MOUSE,                                    # Retain both mice in recording; select only the focal mouse here.
    drop       = True,
)


## Compute Head Position and Head-Forward

In [ ]:
##### Compute Head Position and Head-Forward ############################################

#==== Locate the Ear Midpoint =========
left_ear = position.sel(keypoint='EarL', drop=True)
right_ear = position.sel(keypoint='EarR', drop=True)

head_position = (left_ear + right_ear) / 2                          # Both ears contribute to the head's position.


#==== Calculate Head-Forward ===========
head_forward = movement.kinematics.compute_head_direction_vector(
    data           = position,
    left_keypoint  = 'EarL',
    right_keypoint = 'EarR',
    camera_view    = 'top_down',                                   # Ear labels determine forward in the overhead video.
)


## Select the Moving Target

Use the other mouse's `BodyCenter` position at each acquired video time as the target. Both mice are loaded from the same tracking table. This measures the direction to the other mouse's body centre; no saved ROI is needed.

In [ ]:
##### Select the Other Mouse's Position ################################################

#==== Locate the Moving Target =========
other_position = recording['position'].sel(
    individual = OTHER_MOUSE,
    keypoint   = 'BodyCenter',
    drop       = True,
)


## Compute Egocentric Head Direction

Compute the signed angle from the vector pointing towards the other mouse to the focal mouse's head-forward vector. This preserves movement's ROI-angle convention: in top-down image coordinates, zero is ahead, positive angles are left and negative angles are right. Missing positions or coincident head/target positions remain NaN.

In [ ]:
##### Compute Egocentric Head Direction Using `movement` ################################

#==== Calculate the Target Vector ======
to_other = other_position - head_position                         # One head-to-target vector for each acquired frame.


#==== Calculate the Egocentric Angle ===
egocentric_hd = movement.utils.vector.compute_signed_angle_2d(
    u = to_other,
    v = head_forward,
)

egocentric_hd = egocentric_hd.where(
    (to_other**2).sum('space') > 0,                                # Coincident positions have no defined target direction.
)
egocentric_hd.name = 'egocentric_head_direction'
egocentric_hd.attrs['units'] = 'rad'


#==== Slice to Selected Range ===========
selected_egocentric_hd = slice_stream_per_trial(
    stream     = egocentric_hd,
    trials     = selected_trials,
    segment    = 'trial',
    time_coord = 'time',
)

selected_egocentric_hd = xr.concat(
    objs = selected_egocentric_hd,
    dim  = 'time',
)


## Head Direction Plotting Function

In [ ]:
def add_annotations(ax, 
                    selected_trials, 
                    **kwargs
) -> AnnotationResult:
    """
    Add trial annotations to the supplied time axis.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axis with limits already set in recording seconds.
    selected_trials : pandas.DataFrame
        Trial rows from the same recording as the plotted angles.
    **kwargs
        Additional arguments passed to ``annotate_qc_timeseries``.

    Returns
    -------
    AnnotationResult
        Drawn artists, legend handles and annotation notes.
    """
    return annotate_qc_timeseries(ax=ax, trials=selected_trials, **kwargs)

In [ ]:
##### Plot Egocentric Head Direction on Ax #############################################


def plot_ax_egocentric_head_direction(
    ax: plt.Axes,
    head_direction: xr.DataArray,
    selected_trials: pd.DataFrame | None = None,
    units: str = 'Degrees',
    row_duration: float | None = 60.0,
    subtitle: str | None = 'Egocentric Head Direction',
    add_annotations_func: Callable[..., AnnotationResult] | None = add_annotations,
    annotation_kwargs: dict | None = None,
    **kwargs,
) -> plt.Axes:
    '''
    Plot egocentric head direction angles onto consecutive time rows in the supplied axis.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Empty axis reserved for the plot; the caller sets the figure size.
    head_direction : xarray.DataArray
        One angle per time sample, in radians. Supply a nonempty trace in
        recording-time order, with time in seconds. Missing angles may be NaN.
    selected_trials : pandas.DataFrame or None
        Optional trial rows for the default annotations, not data selection.
        The supplied angle timestamps always determine the plotted range.
    units : str
        Display angles in 'Degrees' or 'Radians'. Input always remains radians.
    row_duration : float or None
        Positive recording seconds per row. None uses one row for the full
        range, or one second for a single timestamp. Last row keeps its width.
    subtitle : str or None
        Title above the first row. None omits the title.
    add_annotations_func : callable or None
        Called as func(row_ax, **annotation_kwargs). The default helper uses
        selected_trials and is skipped when that table is None. A custom
        callback may run without trials. None disables all annotations.
    annotation_kwargs : dict or None
        Arguments for the callback. For the default helper, selected_trials
        is inserted once from this function's argument.
    **kwargs
        Arguments passed to Axes.plot, such as markersize or alpha. Points
        are the default so wrap crossings and excluded intervals stay unjoined.

    Returns
    -------
    matplotlib.axes.Axes
        Original axis; multiple rows are available through ax.child_axes.
    '''
    #==== 0| Validation Checks =======================================
    if ax is None:
        raise ValueError("ax must be provided")
    if head_direction is None:
        raise ValueError("head_direction must be provided")


    #==== 1| Calculate Rows Needed for Subplot =======================
    range_start = float(head_direction['time'][0])
    range_end   = float(head_direction['time'][-1])

    duration = range_end - range_start if row_duration is None else row_duration
    row_count = max(1, int(np.ceil((range_end - range_start) / duration)))



    #==== 2| Create the Required Time Rows ===========================
    if row_count == 1:
        row_axes = [ax]

    else:
        ax.set_axis_off()                                                           # Use the supplied axis as the container for the rows.

        row_axes = []                                                               # List to hold the axes for each row
        row_height = 1 / (row_count + 0.25 * (row_count - 1))                       # Calculate the height of each row considering the gaps between them.
        row_gap = 0.25 * row_height                                                 # Calculate the gap between rows as a fraction of the row height.

        for row in range(row_count):                                                # Create an inset axis for each row
            bottom = 1 - (row + 1) * row_height - row * row_gap                     # Calculate the bottom position of the current row.
            row_axes.append(                                                        # Append the inset axis for the current row to the list of row axes.
                ax.inset_axes([0, bottom, 1, row_height])
            )   


    #==== 3| Convert Units if Necessary ==============================
    if units == "Degrees":
        adjusted_head_direction = np.rad2deg(head_direction)
        limit = 180.0
        ticks = [-180, -90, 0, 90, 180]
        tick_labels = ["-180", "-90", "0", "90", "180"]
    
    elif units == "Radians":
        adjusted_head_direction = head_direction
        limit = np.pi
        ticks = [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi]
        tick_labels = [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"]
    
    else:
        raise ValueError("units must be 'Degrees' or 'Radians'.")



    #==== 4| Set Plotting Parameters =================================
    #       left blank for now intentionally.


    #==== 5| Plot Egocentric HD and Matching Annotations =============
    annotation_kwargs = annotation_kwargs or {'selected_trials': selected_trials} if add_annotations_func is add_annotations else {}

    for row, row_ax in enumerate(row_axes):

        row_start = range_start + row * duration                                    # Calculate start time of current row
        row_end = min(row_start + duration, range_end)                              # Calculate end time of current row, ensure it does not exceed  overall range.

        row_ego_hd = adjusted_head_direction.sel(time=slice(row_start, row_end))

        row_ax.plot(row_ego_hd.time,                                               # Plot the egocentric head direction for the current row
                    row_ego_hd,
                    '.',
                    **kwargs
        )

        row_ax.set(xlim=(row_start, row_start + duration), ylim=(-limit, limit),    # Set the x and y axis limits for the current row
                   xlabel="Time (Seconds)", ylabel=f"Head Direction ({units})")
        row_ax.set_yticks(ticks, labels=tick_labels)                                # Set the y-axis ticks and labels for the current row

        row_ax.axhline(0, linewidth=0.8)                                            # Add a horizontal line at y=0 for reference
       
        # Keep annotation coordinates in the original recording-time units.
        # if add_annotations_func is not None:
        #     if selected_trials is None:
        #         raise ValueError("selected_trials must be provided")

        #     add_annotations_func(
        #         row_ax,
        #         selected_trials,
        #         **(annotation_kwargs or {}),
        #     )
        # Use recording seconds for both the trace and its trial annotations.
        if add_annotations_func is not None:
            add_annotations_func(row_ax, **annotation_kwargs)
    

    #==== 6| Add the Plot Title ======================================
    if subtitle is not None:
        row_axes[0].set_title(subtitle)

    return ax








## Generate Plot

In [ ]:
##### Plot the Selected Egocentric Head Direction ######################################

#==== Size the Figure for the Time Rows =
row_duration = 60.0
plot_duration = float(selected_egocentric_hd.time[-1] - selected_egocentric_hd.time[0])
row_count = max(1, int(np.ceil(plot_duration / row_duration)))

fig, ax = plt.subplots(
    figsize=(13, 2.8 * row_count),                                 # Leave enough height for each row's labels and annotations.
    layout='constrained',
)


#==== Draw Angles and Trial Annotations =
ax = plot_ax_egocentric_head_direction(
    ax                   = ax,
    head_direction       = selected_egocentric_hd,
    selected_trials      = selected_trials,                       # None omits trial annotations; the trace still plots.
    units                = 'Degrees',
    row_duration         = row_duration,
    subtitle             = f'{OTHER_MOUSE} relative to {FOCAL_MOUSE} | Egocentric Head Direction',
    add_annotations_func = add_annotations,                       # Set to None to omit annotations even with trial rows.
    annotation_kwargs    = None,
    markersize           = 2.5,
)

plt.show()